In [1]:
import sys, json
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from src.db import DATA_RAW, MODELS_DIR
from src.features import build_features, get_feature_lists
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor

df = pd.read_parquet(DATA_RAW / "cars.parquet")

# 90% train (финальные модели), 10% calibration (для q_hat)
X_all = df.drop(columns=["price", "seller_description"], errors="ignore")
y_all = np.log1p(df["price"])

X_train, X_cal, y_train, y_cal = train_test_split(
    X_all, y_all, test_size=0.10, random_state=42
)

# определяем фичи ОДИН РАЗ и сохраняем
features, num_f, bool_f, cat_f = get_feature_lists(df)
features = [f for f in features if f in X_train.columns]
cat_f = [f for f in cat_f if f in X_train.columns]

print(f"train: {len(X_train)}, cal: {len(X_cal)}, features: {len(features)}")

# ── обучаем три финальные модели на train ────────────────────────────
models = {}
for q in [0.1, 0.5, 0.9]:
    print(f"\nTraining q={q}...")
    m = CatBoostRegressor(
        loss_function=f"Quantile:alpha={q}",
        iterations=3000, learning_rate=0.05, depth=8,
        cat_features=cat_f, random_seed=42, verbose=500,
    )
    m.fit(X_train[features], y_train)
    models[q] = m
    m.save_model(MODELS_DIR / f"catboost_final_q{int(q*100)}.cbm")
    print(f"[saved] catboost_final_q{int(q*100)}.cbm")

# ── считаем q_hat на калибровочном наборе ────────────────────────────
q10_cal = models[0.1].predict(X_cal[features])
q90_cal = models[0.9].predict(X_cal[features])
scores = np.maximum(q10_cal - y_cal.values, y_cal.values - q90_cal)
n = len(scores)
level = min(np.ceil((n + 1) * 0.80) / n, 1.0)
q_hat = float(np.quantile(scores, level))
print(f"\nq_hat = {q_hat:.5f}")

# ── проверка coverage на calibration ─────────────────────────────────
y_cal_rub = np.expm1(y_cal)
q10_cp = np.expm1(q10_cal - q_hat)
q90_cp = np.expm1(q90_cal + q_hat)
cov = ((y_cal_rub >= q10_cp) & (y_cal_rub <= q90_cp)).mean()
print(f"Coverage on cal: {cov:.3f} (цель ≈ 0.80)")

# ── сохраняем метаданные ─────────────────────────────────────────────
(MODELS_DIR / "q_hat.txt").write_text(str(q_hat))
(MODELS_DIR / "features.json").write_text(json.dumps(features))
(MODELS_DIR / "cat_features.json").write_text(json.dumps(cat_f))
print(f"[saved] features.json, cat_features.json, q_hat.txt")

train: 27136, cal: 3016, features: 68

Training q=0.1...
0:	learn: 0.1525477	total: 200ms	remaining: 9m 59s
500:	learn: 0.0292800	total: 21.5s	remaining: 1m 47s
1000:	learn: 0.0274018	total: 42.1s	remaining: 1m 24s
1500:	learn: 0.0267893	total: 1m 2s	remaining: 1m 2s
2000:	learn: 0.0263913	total: 1m 22s	remaining: 41.3s
2500:	learn: 0.0259146	total: 1m 42s	remaining: 20.5s
2999:	learn: 0.0256216	total: 2m 3s	remaining: 0us
[saved] catboost_final_q10.cbm

Training q=0.5...
0:	learn: 0.3377599	total: 40.5ms	remaining: 2m 1s
500:	learn: 0.0595497	total: 20.9s	remaining: 1m 44s
1000:	learn: 0.0512410	total: 42.1s	remaining: 1m 24s
1500:	learn: 0.0473452	total: 1m 3s	remaining: 1m 3s
2000:	learn: 0.0448208	total: 1m 25s	remaining: 42.5s
2500:	learn: 0.0429542	total: 1m 46s	remaining: 21.3s
2999:	learn: 0.0413770	total: 2m 7s	remaining: 0us
[saved] catboost_final_q50.cbm

Training q=0.9...
0:	learn: 0.1613795	total: 41.7ms	remaining: 2m 5s
500:	learn: 0.0260118	total: 21.4s	remaining: 1m 46s